# Advanced Graph Algorithms

Notes and runnable examples on the graph algorithms *beyond* shortest paths and spanning trees. The **graphs** notebook gave us representations and **BFS/DFS**; **graph algorithms** added **Dijkstra**, **Bellman-Ford**, **A\***, **union-find**, and **MST**. This notebook covers the structural and combinatorial heavyweights: **strongly connected components**, **bridges & articulation points**, **max-flow / min-cut**, and **bipartite matching**. Each is still a `dict` graph plus a stack, a `deque`, or a `set` — wired a little more cleverly.

Every claim below is backed by a runnable proof on a small graph whose answer you can check by hand: SCC partitions, named bridges and cut-vertices, the max-flow–min-cut theorem, and valid matchings (with König's theorem as a bonus).

**Contents**

1. **The landscape** — what these solve
2. **Strongly connected components** — Kosaraju & Tarjan
3. **Bridges & articulation points** — disc/low timestamps
4. **Max-flow / min-cut** — Edmonds-Karp
5. **Bipartite matching** — Kuhn's algorithm (+ König)
6. **When to use what**
7. **Complexity cheat-sheet**

## 1. The landscape

The **graphs** notebook handed us the structure and the traversals; **graph algorithms** answered *"what's the cheapest route?"* (Dijkstra, Bellman-Ford, A\*) and *"what's the cheapest way to connect everything?"* (Kruskal, Prim). Those are the **weight/optimization** questions. This notebook answers a different family — questions about a graph's **structure** and its **capacity**:

| Question | Algorithm | Built from |
|---|---|---|
| Which vertices can reach each other (directed)? | **Kosaraju / Tarjan** SCC | two DFS passes / one DFS + a stack |
| Which edges/vertices, if cut, disconnect the graph? | **bridges & articulation points** | one DFS + disc/low timestamps |
| How much can flow from source to sink? | **Edmonds-Karp** max-flow | BFS over a residual graph |
| What's the largest set of disjoint pairings? | **Kuhn** bipartite matching | DFS augmenting paths |

The throughline this time is **DFS timestamps and augmenting paths**. SCCs and bridges both fall out of *when* DFS first sees a vertex versus the *earliest* vertex it can climb back to; flow and matching both work by *finding a path that improves the current solution, then flipping it*. Two ideas, four algorithms.

> A note on depth: SCC and bridge algorithms are textbook-recursive, but CPython's default recursion limit (1000, raised to 3000 under Jupyter — see **recursion & backtracking**) makes deep recursion fragile. The from-scratch implementations here are **iterative** (an explicit stack) so they'd survive a million-vertex chain; the recursive Kuhn demo is kept deliberately tiny.

## 2. Strongly connected components

In a **directed** graph, a **strongly connected component (SCC)** is a maximal set of vertices where every vertex can reach every other. Contracting each SCC to a point yields the **condensation** — always a DAG. SCCs are the directed analogue of connected components (for undirected graphs, plain **BFS/DFS** suffices); they answer *"are these two states mutually reachable?"* and underpin 2-SAT, dependency-cycle detection, and dead-code analysis.

Our test digraph has three obvious SCCs — two 3-cycles and a lone sink:

```
  0 → 1 → 2 → 0      (cycle A = {0,1,2})
          2 → 3
  3 → 4 → 5 → 3      (cycle B = {3,4,5})
          5 → 6      (6 = lone sink)
```

In [1]:
# Directed test graph: SCCs are {0,1,2}, {3,4,5}, {6}  (verifiable by eye)
digraph = {
    0: [1], 1: [2], 2: [0, 3],
    3: [4], 4: [5], 5: [3, 6],
    6: [],
}
EXPECTED = {frozenset({0, 1, 2}), frozenset({3, 4, 5}), frozenset({6})}

### Kosaraju — two DFS passes

**Kosaraju's algorithm** is the easiest to *believe*. Three steps:

1. **DFS the graph**, pushing each vertex onto an `order` list **when it finishes** (all descendants done). This is a reverse-topological order of the condensation.
2. **Transpose** the graph (reverse every edge).
3. **DFS the transpose** in *reverse finish order*. Each tree you grow is exactly one SCC — you "peel" them off sink-component first.

Why it works: finishing-time order guarantees the *last-finished* vertex lies in a **source** SCC of the condensation. Reversing edges turns that source into a sink, so a DFS from it can't leak into any other component — it captures precisely its own SCC, then the next, and so on. Both passes are **iterative** here (an explicit stack with a child-iterator) to dodge the recursion limit.

In [2]:
def scc_kosaraju(graph):
    # Pass 1: iterative DFS, append each vertex when it FINISHES.
    order, seen = [], set()
    for s in graph:
        if s in seen:
            continue
        seen.add(s)
        stack = [(s, iter(graph[s]))]            # (vertex, neighbor-iterator)
        while stack:
            node, it = stack[-1]
            for v in it:                          # advance to first unseen child
                if v not in seen:
                    seen.add(v)
                    stack.append((v, iter(graph[v])))
                    break
            else:                                 # iterator exhausted -> node finished
                order.append(node)
                stack.pop()

    # Pass 2: transpose (reverse every edge).
    t = {u: [] for u in graph}
    for u in graph:
        for v in graph[u]:
            t[v].append(u)

    # Pass 3: DFS the transpose in REVERSE finish order; each tree is one SCC.
    comps, seen2 = [], set()
    for s in reversed(order):
        if s in seen2:
            continue
        comp, stack = [], [s]
        seen2.add(s)
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in t[u]:
                if v not in seen2:
                    seen2.add(v)
                    stack.append(v)
        comps.append(frozenset(comp))
    return comps

kos = scc_kosaraju(digraph)
print("Kosaraju SCCs:", sorted(sorted(c) for c in kos))
assert set(kos) == EXPECTED
print("matches the hand-checked partition ->", set(kos) == EXPECTED)

Kosaraju SCCs: [[0, 1, 2], [3, 4, 5], [6]]
matches the hand-checked partition -> True


### Tarjan — one DFS, disc/low, a stack

**Tarjan's algorithm** gets the same answer in a **single** DFS — no transpose, no second pass. It assigns each vertex a discovery index `disc[v]` (the order DFS first reached it) and a `low[v]` = the smallest `disc` reachable from `v`'s subtree using **tree edges plus at most one back-edge to a vertex still on the stack**. It keeps all vertices of the current path on an explicit `stack`. When a vertex `v` satisfies `low[v] == disc[v]`, it's the **root** of an SCC: everything above it on the stack (down to and including `v`) forms that component.

The `on_stack` set is the subtle part — you only relax `low` against a neighbor that's *still on the stack*, because a neighbor already popped into a finished SCC isn't reachable *back*. Again iterative, with the parent's `low` updated as each child unwinds (the line a recursive version writes as `low[u] = min(low[u], low[child])`).

In [3]:
def scc_tarjan(graph):
    disc, low = {}, {}
    on_stack, stack, comps = set(), [], []
    counter = 0
    for start in graph:
        if start in disc:
            continue
        disc[start] = low[start] = counter; counter += 1
        stack.append(start); on_stack.add(start)
        work = [(start, iter(graph[start]))]      # explicit DFS stack
        while work:
            node, it = work[-1]
            descended = False
            for w in it:
                if w not in disc:                 # tree edge -> descend
                    disc[w] = low[w] = counter; counter += 1
                    stack.append(w); on_stack.add(w)
                    work.append((w, iter(graph[w])))
                    descended = True
                    break
                elif w in on_stack:               # back-edge to a live vertex
                    low[node] = min(low[node], disc[w])
            if descended:
                continue
            if low[node] == disc[node]:           # node roots an SCC -> peel it off
                comp = []
                while True:
                    w = stack.pop(); on_stack.discard(w)
                    comp.append(w)
                    if w == node:
                        break
                comps.append(frozenset(comp))
            work.pop()
            if work:                              # propagate low up to the parent
                low[work[-1][0]] = min(low[work[-1][0]], low[node])
    return comps

tar = scc_tarjan(digraph)
print("Tarjan SCCs  :", sorted(sorted(c) for c in tar))
assert set(tar) == EXPECTED
assert set(tar) == set(scc_kosaraju(digraph))      # two independent algorithms agree
print("Tarjan == Kosaraju == hand-checked ->", set(tar) == EXPECTED)

Tarjan SCCs  : [[0, 1, 2], [3, 4, 5], [6]]
Tarjan == Kosaraju == hand-checked -> True


## 3. Bridges & articulation points

In an **undirected** graph, a **bridge** is an edge whose removal increases the number of connected components, and an **articulation point** (cut vertex) is a vertex whose removal does the same. They're the single points of failure in a network — the links and routers whose loss splits it in two.

Both fall out of one DFS using the same `disc`/`low` timestamps as Tarjan. For a tree edge `u—v` (with `v` the child):

- **Bridge** when `low[v] > disc[u]` — `v`'s subtree has **no** back-edge reaching `u` or above, so the only way in is through this edge.
- **Articulation point** (non-root `u`) when `low[v] >= disc[u]` — `v`'s subtree can climb no higher than `u`, so removing `u` strands it. The **root** is special: it's a cut vertex iff it has **two or more** DFS children.

Test graph: a triangle `0-1-2` hanging off a tail `2-3-4`.

```
    0 --- 1
     \   /
      2 ---- 3 ---- 4
```

By eye: edges `2-3` and `3-4` are **bridges** (the triangle's internal edges aren't — each has an alternate route). Vertices `2` and `3` are **cut vertices** (lose 2 and {3,4} fall off; lose 3 and 4 falls off).

In [4]:
def bridges_and_cuts(graph):
    disc, low = {}, {}
    timer = 0
    bridges, cuts = [], set()
    for start in graph:
        if start in disc:
            continue
        disc[start] = low[start] = timer; timer += 1
        root, root_kids = start, 0
        stack = [(start, None, iter(graph[start]))]   # (vertex, parent, neighbors)
        while stack:
            node, parent, it = stack[-1]
            descended = False
            for v in it:
                if v == parent:
                    continue                          # don't bounce back up the tree edge
                if v not in disc:                     # tree edge -> descend
                    if node == root:
                        root_kids += 1
                    disc[v] = low[v] = timer; timer += 1
                    stack.append((v, node, iter(graph[v])))
                    descended = True
                    break
                else:                                 # back-edge
                    low[node] = min(low[node], disc[v])
            if descended:
                continue
            stack.pop()
            if stack:                                 # unwinding: update parent p
                p = stack[-1][0]
                low[p] = min(low[p], low[node])
                if p != root and low[node] >= disc[p]:
                    cuts.add(p)                       # non-root cut vertex
                if low[node] > disc[p]:
                    bridges.append(tuple(sorted((p, node))))
        if root_kids > 1:
            cuts.add(root)                            # root with >=2 children
    return sorted(set(bridges)), cuts

undirected = {
    0: [1, 2],
    1: [0, 2],
    2: [0, 1, 3],
    3: [2, 4],
    4: [3],
}
br, cuts = bridges_and_cuts(undirected)
print("bridges       :", br)
print("articulation  :", sorted(cuts))
assert br == [(2, 3), (3, 4)]                          # the two tail edges, by hand
assert cuts == {2, 3}                                  # the two cut vertices, by hand
print("bridges & cut-vertices match the hand-checked graph ->", br == [(2, 3), (3, 4)] and cuts == {2, 3})

bridges       : [(2, 3), (3, 4)]
articulation  : [2, 3]
bridges & cut-vertices match the hand-checked graph -> True


## 4. Max-flow / min-cut

Give every edge a **capacity** and ask: how much can flow from a **source** `s` to a **sink** `t` without exceeding any edge? That's the **maximum-flow** problem. **Ford-Fulkerson** solves it by repeatedly finding an **augmenting path** — any `s→t` path with spare capacity — and pushing as much flow as its tightest edge allows. The trick that makes it correct is the **residual graph**: pushing `f` along `u→v` subtracts `f` from that edge *and adds `f` to a reverse edge* `v→u`, leaving the option to *cancel* flow later.

**Edmonds-Karp** is Ford-Fulkerson that always picks the **shortest** augmenting path (BFS, fewest edges). That single choice bounds it to **O(V·E²)** regardless of capacities — plain Ford-Fulkerson with bad path choices can be exponential.

The payoff is the **max-flow–min-cut theorem**: the maximum flow equals the capacity of the minimum `s`-`t` **cut** (the cheapest set of edges to sever so `s` can't reach `t`). And the min cut falls out for free — after the algorithm halts, the set of vertices still **reachable from `s` in the residual graph** is one side of a min cut. We'll verify the equality on the classic CLRS network (answer: **23**).

In [5]:
from collections import deque, defaultdict

def edmonds_karp(capacity, s, t):
    res = defaultdict(lambda: defaultdict(int))    # residual capacities
    for u in capacity:
        for v, c in capacity[u].items():
            res[u][v] += c
            res[v][u] += 0                         # make sure reverse edge exists (0 cap)
    flow = 0
    while True:
        parent = {s: None}                          # BFS for a SHORTEST augmenting path
        q = deque([s])
        while q and t not in parent:
            u = q.popleft()
            for v in res[u]:
                if v not in parent and res[u][v] > 0:
                    parent[v] = u
                    q.append(v)
        if t not in parent:
            break                                   # no augmenting path -> done
        bottleneck = float("inf")                   # tightest residual edge on the path
        v = t
        while parent[v] is not None:
            bottleneck = min(bottleneck, res[parent[v]][v]); v = parent[v]
        v = t                                       # push flow, opening reverse edges
        while parent[v] is not None:
            u = parent[v]
            res[u][v] -= bottleneck
            res[v][u] += bottleneck
            v = u
        flow += bottleneck

    # Min cut: vertices still reachable from s in the final residual graph.
    reach, q = {s}, deque([s])
    while q:
        u = q.popleft()
        for v in res[u]:
            if v not in reach and res[u][v] > 0:
                reach.add(v); q.append(v)
    cut_cap, cut_edges = 0, []
    for u in capacity:
        for v, c in capacity[u].items():
            if u in reach and v not in reach:       # original edge crossing the cut
                cut_cap += c
                cut_edges.append((u, v, c))
    return flow, reach, cut_cap, cut_edges

# Classic CLRS network; known max flow = 23.
capacity = {
    "s": {"a": 16, "b": 13},
    "a": {"c": 12},
    "b": {"a": 4, "d": 14},
    "c": {"b": 9, "t": 20},
    "d": {"c": 7, "t": 4},
}
flow, reach, cut_cap, cut_edges = edmonds_karp(capacity, "s", "t")
print("max flow            :", flow)
print("min-cut S-side      :", sorted(reach))
print("min-cut edges       :", cut_edges)
print("min-cut capacity    :", cut_cap)
assert flow == 23                                   # the known answer
assert cut_cap == flow                              # max-flow == min-cut, proved
print("max-flow == min-cut ->", flow == cut_cap == 23)

max flow            : 23
min-cut S-side      : ['a', 'b', 'd', 's']
min-cut edges       : [('a', 'c', 12), ('d', 'c', 7), ('d', 't', 4)]
min-cut capacity    : 23
max-flow == min-cut -> True


## 5. Bipartite matching — Kuhn's algorithm

A **bipartite graph** splits into two sides (workers/jobs, applicants/slots) with edges only *across*. A **matching** picks edges so no vertex is used twice; a **maximum matching** picks as many as possible. This is a **special case of max-flow** — add a source to every left vertex and every right vertex to a sink, all capacities 1, and the max flow *is* the maximum matching — but **Kuhn's algorithm** does it directly and is simpler to read.

Kuhn grows the matching one vertex at a time via **augmenting paths**: for each left vertex `u`, DFS for an unused right neighbor, *or* a right neighbor whose current partner can be **rebumped** to someone else. Each successful DFS flips the path and grows the matching by exactly one. It's the same augmenting-path idea as max-flow, stripped to the bipartite case — **O(V·E)**. This demo is small, so the recursive DFS is safe (deep graphs would need the iterative style from §2–3, or convert to flow).

Test graph (perfect matching exists): left `{0,1,2,3}`, right `{a,b,c,d}`.

In [6]:
def kuhn(adj, left):
    match_r = {}                                    # right vertex -> its matched left vertex
    def augment(u, visited):
        for v in adj.get(u, []):
            if v in visited:
                continue
            visited.add(v)
            # v is free, OR v's partner can be rematched elsewhere -> steal v
            if v not in match_r or augment(match_r[v], visited):
                match_r[v] = u
                return True
        return False
    for u in left:
        augment(u, set())
    return match_r

adj = {
    0: ["a", "b"],
    1: ["a"],
    2: ["b", "c"],
    3: ["c", "d"],
}
left = [0, 1, 2, 3]
match_r = kuhn(adj, left)
print("matching (right->left):", match_r)
print("matching size         :", len(match_r))

# Validity: every matched edge exists, and no left vertex is reused.
for v, u in match_r.items():
    assert v in adj[u]                              # it's a real edge
assert len(set(match_r.values())) == len(match_r)  # each left used at most once
assert len(match_r) == 4                            # perfect matching, by hand
print("valid perfect matching of size 4 ->", len(match_r) == 4)

matching (right->left): {'a': 1, 'b': 0, 'c': 2, 'd': 3}
matching size         : 4
valid perfect matching of size 4 -> True


### König's theorem — matching meets cover

**König's theorem**: in a bipartite graph, the size of a **maximum matching** equals the size of a **minimum vertex cover** (the fewest vertices that touch every edge). It's a cornerstone duality — the matching version of max-flow–min-cut. The minimum cover is *constructible* from a maximum matching: run an alternating BFS from the **unmatched left** vertices, then take **`(Left \ reached) ∪ (Right ∩ reached)`**.

Here's a graph with **5** left and **3** right vertices, so the matching can't be perfect — max matching is 3, and we verify a 3-vertex cover exists:

In [7]:
# 5 left, 3 right -> matching capped at 3 (a non-trivial cover)
adj2 = {0: ["a", "b"], 1: ["a"], 2: ["b", "c"], 3: ["c"], 4: ["a", "c"]}
left2, right2 = [0, 1, 2, 3, 4], ["a", "b", "c"]
match_r2 = kuhn(adj2, left2)
match_l2 = {u: v for v, u in match_r2.items()}
M = len(match_r2)
print("max matching size:", M)

# Konig construction: alternating BFS from unmatched-left vertices.
reached_L = {u for u in left2 if u not in match_l2}
reached_R = set()
dq = deque((u, "L") for u in reached_L)
while dq:
    node, side = dq.popleft()
    if side == "L":                                 # follow NON-matching edges L->R
        for v in adj2[node]:
            if match_l2.get(node) != v and v not in reached_R:
                reached_R.add(v); dq.append((v, "R"))
    elif node in match_r2:                           # follow the MATCHING edge R->L
        u = match_r2[node]
        if u not in reached_L:
            reached_L.add(u); dq.append((u, "L"))

cover = (set(left2) - reached_L) | (set(right2) & reached_R)
print("min vertex cover :", sorted(map(str, cover)), "(size", len(cover), ")")
assert len(cover) == M                              # Konig: |cover| == |matching|
for u in adj2:                                       # the cover really covers every edge
    for v in adj2[u]:
        assert u in cover or v in cover
print("Konig verified: |min vertex cover| == |max matching| ==", M)

max matching size: 3
min vertex cover : ['a', 'b', 'c'] (size 3 )
Konig verified: |min vertex cover| == |max matching| == 3


## 6. When to use what

| You need... | Reach for | Notes |
|---|---|---|
| Components of an **undirected** graph | BFS/DFS flood (**graphs** notebook) | plain traversal, O(V+E) |
| Components of a **directed** graph (mutual reachability) | **Kosaraju** or **Tarjan** | Tarjan is one pass; Kosaraju is easier to reason about |
| Cycle detection in a digraph | **SCC** (any SCC of size >1, or a self-loop) | or DFS color-marking in **graphs** |
| 2-SAT / implication solving | **SCC** on the implication graph | x and ¬x in the same SCC ⇒ unsatisfiable |
| Single points of failure (links) | **bridges** | network reliability, biconnectivity |
| Single points of failure (routers) | **articulation points** | same DFS, `>=` instead of `>` |
| Throughput / capacity from a source to a sink | **Edmonds-Karp** max-flow | shortest augmenting paths, O(V·E²) |
| Cheapest way to disconnect s from t | **min cut** | falls out of the final residual graph |
| Largest set of disjoint pairings | **Kuhn** bipartite matching | a flow special case; O(V·E) |
| Fewest vertices covering all edges (bipartite) | **min vertex cover** via **König** | = max matching size |

All four sit on top of the same primitives as the rest of the series — a `dict` graph, a `deque` or explicit stack, and a `set`. The cleverness is in *what order you explore* and *what you flip when you find a path*, not in any exotic data structure.

## 7. Complexity cheat-sheet

| Algorithm | Solves | Time | Space |
|---|---|:---:|:---:|
| **Kosaraju** SCC | strongly connected components | O(V + E) | O(V + E) (transpose) |
| **Tarjan** SCC | strongly connected components | O(V + E) | O(V) (one DFS + stack) |
| **Bridges / articulation** | edges/vertices that disconnect | O(V + E) | O(V) |
| **Edmonds-Karp** | max-flow / min-cut | O(V · E²) | O(V + E) (residual) |
| **Kuhn** | maximum bipartite matching | O(V · E) | O(V + E) |
| **Dinic** (not shown) | max-flow, faster | O(V² · E) | O(V + E) |
| **Hopcroft–Karp** (not shown) | bipartite matching, faster | O(E·√V) | O(V + E) |

<sub>V = vertices, E = edges. The from-scratch SCC and bridge code here is **iterative** to survive deep graphs (CPython's recursion limit — see **recursion & backtracking**). For dense flow problems reach for **Dinic**; for large bipartite matching, **Hopcroft–Karp**. Shortest-path and MST algorithms live in the **graph algorithms** notebook; representations and **BFS/DFS** in the **graphs** notebook.</sub>

---

**Where this fits.** With **graphs** (structure + traversals), **graph algorithms** (shortest paths + MST), and this notebook (connectivity structure + flow + matching), the graph track is complete. The recurring lesson holds: the headline algorithms — Tarjan, Edmonds-Karp, Kuhn — are just **DFS timestamps** and **augmenting paths** layered on a `dict`-of-lists, a `deque`, and a `set`. Pick the right traversal order, flip the right path, and the structure does the rest.